In [1]:
import requests
import json
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

In [2]:
def safe_json(r, label=""):
    """Parsea JSON de una respuesta requests; imprime error y devuelve None si falla."""
    try:
        return r.json()
    except Exception:
        print(f"[ERROR json] {label} — HTTP {r.status_code} — body: {r.text[:300]!r}")
        return None


def load_geojson_jsonp(url):
    """Descarga un GeoJSON en formato JSONP y devuelve la lista de features."""
    response = requests.get(url)
    response.raise_for_status()
    raw = response.text.strip()
    if raw.startswith("jsonCallback("):
        raw = raw[len("jsonCallback("):]
        if raw.endswith(");"):
            raw = raw[:-2]
        elif raw.endswith(")"):
            raw = raw[:-1]
    return json.loads(raw)['features']


def features_to_df(features):
    """Convierte la lista de features GeoJSON a un DataFrame con las columnas de interés."""
    df = pd.DataFrame([{
        "nombre":      f['properties'].get('documentname'),
        "descripcion": f['properties'].get('documentdescription') or "",
        "municipio":   f['properties'].get('municipality'),
        "provincia":   f['properties'].get('territory'),
        "latitud":     f['geometry']['coordinates'][1],
        "longitud":    f['geometry']['coordinates'][0],
        "type":        f['properties'].get('templatetype'),
        "tipo-comida": f['properties'].get('restorationtype'),
        "entorno":     f['properties'].get('marks'),
        "quality":     f['properties'].get('qualityassurance'),
        "euskal_sello": f['properties'].get('productclub'),
        "email":       f['properties'].get('tourismemail') or "",
        "web":         f['properties'].get('web') or "",
        "web_euskadi": f['properties'].get('friendlyurl') or ""
    } for f in features])
    # Normalizar URLs a https para evitar redirecciones
    df["web"] = df["web"].str.replace("http:", "https:", regex=False)
    df["web_euskadi"] = df["web_euskadi"].str.replace("http:", "https:", regex=False)
    return df


# --- Ejecución ---
URL_GEOJSON = "https://opendata.euskadi.eus/contenidos/ds_recursos_turisticos/restaurantes_sidrerias_bodegas/opendata/restaurantes-padding.geojson"

features = load_geojson_jsonp(URL_GEOJSON)
print(f"[OK] GeoJSON descargado: {len(features)} restaurantes")

df_opendata_restaurantes = features_to_df(features)



[OK] GeoJSON descargado: 716 restaurantes


In [ ]:
def merge_dataframes(df1, df2, cat1, cat2):
    """Une df1 (cat1) con df2 (cat2).
    Conserva todas las filas de df1 y añade de df2 solo las que no existan en df1
    (comparando por nombre + municipio). Si existe duplicado, actualiza la categoria de df1."""
    df1 = df1.copy()
    df2 = df2.copy()

    # Solo inicializar categoria en df1 si no existe (para no sobreescribir merges previos)
    if "categoria" not in df1.columns:
        df1["categoria"] = cat1
    df2["categoria"] = cat2

    # Filas de df2 cuyo (nombre, municipio) NO está en df1
    key1 = set(zip(df1["nombre"], df1["municipio"]))
    mask_duplicados = df2.apply(lambda r: (r["nombre"], r["municipio"]) in key1, axis=1)
    nuevos = df2[~mask_duplicados]
    duplicados = df2[mask_duplicados]

    if len(duplicados) > 0:
        print(f"Duplicados de {cat2} ya existentes en df1: {len(duplicados)}")
        print(duplicados[["nombre", "municipio"]].to_string(index=False))

        # Actualizar la categoria de df1 para las filas duplicadas
        dup_keys = set(zip(duplicados["nombre"], duplicados["municipio"]))
        mask_en_df1 = df1.apply(lambda r: (r["nombre"], r["municipio"]) in dup_keys, axis=1)
        df1.loc[mask_en_df1, "categoria"] = df1.loc[mask_en_df1, "categoria"] + ", " + cat2

    result = pd.concat([df1, nuevos], ignore_index=True)
    print(f"Total tras merge: {len(result)} filas ({cat1}: {len(df1)}, nuevos {cat2}: {len(nuevos)})")
    return result


In [4]:
URL_GEOJSON_QUESERIA = "https://opendata.euskadi.eus/contenidos/ds_recursos_turisticos/queserias_conservera_productor/opendata/queserias-conserveras-productores-padding.geojson"
features_queseria = load_geojson_jsonp(URL_GEOJSON_QUESERIA)
df_opendata_queseria = features_to_df(features_queseria)
print(f"[OK] GeoJSON descargado: {len(features_queseria)} queserías")




[OK] GeoJSON descargado: 58 queserías


In [5]:
df_opendata = merge_dataframes(df_opendata_restaurantes, df_opendata_queseria, "restaurantes", "queseria")

Total tras merge: 774 filas (restaurantes: 716, queseria: 58)


In [6]:
URL_GEOJSON_BODEGA = "https://opendata.euskadi.eus/contenidos/ds_recursos_turisticos/bodegas_vino_txakoli/opendata/bodegas.geojson"
features_bodega = load_geojson_jsonp(URL_GEOJSON_BODEGA)
df_opendata_bodega = features_to_df(features_bodega)
print(f"[OK] GeoJSON descargado: {len(features_bodega)} bodegas")

[OK] GeoJSON descargado: 104 bodegas


In [7]:
df_opendata = merge_dataframes(df_opendata, df_opendata_bodega, "restaurantes", "bodegas")

Duplicados ignorados de bodegas: 104
                                nombre                      municipio
                               Ameztoi                        Getaria
               Arabako Txakolina, S.L.                        Amurrio
                      Beldui Txakolina                  Laudio/Llodio
                               Berroja                         Muxika
                        Bodegas Tierra              Labastida/Bastida
                 Bodegas Amador García          Baños de Ebro/Mañueta
                              Baigorri                      Samaniego
                      Bodegas Campillo                      Laguardia
                                Covila            Lapuebla de Labarca
                     Dominio de Berzal          Baños de Ebro/Mañueta
                          El Fabulista                      Laguardia
                          Espada Ojeda            Lapuebla de Labarca
                             Estraunza            Lap

In [8]:
# seleccionamos los distintos tipos de "tipo-comida"
print("Tipos de comida disponibles:")
df_opendata["tipo-comida"].unique()


Tipos de comida disponibles:


array(['Restaurante', 'Bodega Txakoli', 'Asador', 'Bodega', 'Sidreria',
       None, 'Trujal', 'Pastelería / Confitería', 'Comida Rápida'],
      dtype=object)

In [9]:
# eliminar las con tipo-comida "comida rápida"
df_opendata = df_opendata[df_opendata["tipo-comida"] != "Comida Rápida"].copy()


In [10]:
df_tiendas = pd.read_csv("data/zonas_compras_limpio.csv")
# rename columnas para cohincidir con el dataset de restaurantes df_opendata
df_tiendas = df_tiendas.rename(columns={"documentname": "nombre", "documentdescription": "descripcion", "municipality": "municipio", "territory": "provincia", "templatetype": "type", "restorationtype": "tipo-comida", "marks": "entorno", "qualityassurance": "quality", "productclub": "euskal_sello", "tourismemail": "email", "web": "web", "friendlyurl": "web_euskadi", "lat": "latitud", "lon": "longitud"})
#seleccionamos las filas cuyo buytype = "Eusko Label", "Agricultura Ecológica", "Euskal Baserri", 'Denominación de Origen'
df_tiendas = df_tiendas[df_tiendas["buytype"].isin(["Eusko Label", "Agricultura Ecológica", "Euskal Baserri", 'Denominación de Origen'])].copy()
df_opendata = merge_dataframes(df_opendata, df_tiendas, "restaurantes", "tiendas")

Duplicados ignorados de tiendas: 58
                                        nombre                          municipio
                                Ezkur - Txerri                            Getaria
                                         Garoa                             Zerain
                                  Gaztaindizar                            Zarautz
                               Mahala Naturala                            Leaburu
                                  Gaztiñatxulo                          Idiazabal
                             Germinados Kimuak                      Maruri-Jatabe
                            Gomiztegi Baserria                              Oñati
                                        MAISOR                            Getaria
                                        Jakion                     Leintz Gatzaga
                     Lactea De Balmaseda, S.L.                          Balmaseda
                                         Borda                

In [16]:
df_opendata

,nombre,descripcion,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,quality,euskal_sello,email,web,web_euskadi,categoria
0,Agorregi,"El restaurante Agorregi, ubicado en el barrio ...",Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",None,None,agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
1,Aizian,Este moderno y acogedor restaurante fue diseña...,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,None,None,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
2,Akelarre,Pedro Subijana desarrolla desde 1970 una cocin...,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,None,1,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
3,Alameda,La tercera generación es quien está a cargo de...,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,None,1,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
4,Almiketxu,Es un caserío típico vasco y está regentado po...,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,None,1,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
784,Ibáñez-Gozona,NaN,Tolosa,Gipuzkoa,43.139897,-2.073319,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas
785,Saizar,NaN,Tolosa,Gipuzkoa,43.132487,-2.080295,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas
786,Suna Gozotegia,NaN,Getaria,Gipuzkoa,43.302922,-2.205573,Denominación de Origen,NaN,Costa Vasca,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas
787,Rafa Gorrotxategi,NaN,Tolosa,Gipuzkoa,43.126214,-2.083494,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,info@rafagorrotxategi.eus,https://rafagorrotxategi.eus/museum/,https://turismoa.euskadi.eus/es/tiendas-gourme...,tiendas


In [12]:
# eliminamos la columna locality
df_opendata = df_opendata.drop(columns=["locality"], errors="ignore")

In [13]:
# los valores de la columna buytype los volcamos a type y eliminamos la columna buytype
df_opendata["type"] = np.where(df_opendata["categoria"] == "tiendas", df_opendata["buytype"], df_opendata["type"])
df_opendata = df_opendata.drop(columns=["buytype"], errors="ignore")

In [15]:
df_opendata

,nombre,descripcion,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,quality,euskal_sello,email,web,web_euskadi,categoria
0,Agorregi,"El restaurante Agorregi, ubicado en el barrio ...",Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",None,None,agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
1,Aizian,Este moderno y acogedor restaurante fue diseña...,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,None,None,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
2,Akelarre,Pedro Subijana desarrolla desde 1970 una cocin...,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,None,1,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
3,Alameda,La tercera generación es quien está a cargo de...,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,None,1,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
4,Almiketxu,Es un caserío típico vasco y está regentado po...,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,None,1,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
784,Ibáñez-Gozona,NaN,Tolosa,Gipuzkoa,43.139897,-2.073319,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas
785,Saizar,NaN,Tolosa,Gipuzkoa,43.132487,-2.080295,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas
786,Suna Gozotegia,NaN,Getaria,Gipuzkoa,43.302922,-2.205573,Denominación de Origen,NaN,Costa Vasca,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas
787,Rafa Gorrotxategi,NaN,Tolosa,Gipuzkoa,43.126214,-2.083494,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,info@rafagorrotxategi.eus,https://rafagorrotxategi.eus/museum/,https://turismoa.euskadi.eus/es/tiendas-gourme...,tiendas


In [99]:
# calidad = True si el lugar tiene al menos un sello de calidad o producto Euskal
def combinar_calidad(row):
    """Devuelve True si el lugar tiene algún sello de calidad o Euskal produktua."""
    return bool(row["quality"]) or bool(row["euskal_sello"])

df_opendata["calidad"] = df_opendata.apply(combinar_calidad, axis=1)


In [100]:
df_opendata

,nombre,descripcion,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,quality,euskal_sello,email,web,web_euskadi,categoria,calidad
0,Agorregi,"El restaurante Agorregi, ubicado en el barrio ...",Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",None,None,agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False
1,Aizian,Este moderno y acogedor restaurante fue diseña...,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,None,None,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False
2,Akelarre,Pedro Subijana desarrolla desde 1970 una cocin...,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,None,1,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
3,Alameda,La tercera generación es quien está a cargo de...,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,None,1,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
4,Almiketxu,Es un caserío típico vasco y está regentado po...,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,None,1,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
784,Ibáñez-Gozona,NaN,Tolosa,Gipuzkoa,43.139897,-2.073319,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True
785,Saizar,NaN,Tolosa,Gipuzkoa,43.132487,-2.080295,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True
786,Suna Gozotegia,NaN,Getaria,Gipuzkoa,43.302922,-2.205573,Denominación de Origen,NaN,Costa Vasca,NaN,1.0,NaN,NaN,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True
787,Rafa Gorrotxategi,NaN,Tolosa,Gipuzkoa,43.126214,-2.083494,Denominación de Origen,NaN,Montes y Valles vascos,NaN,1.0,info@rafagorrotxategi.eus,https://rafagorrotxategi.eus/museum/,https://turismoa.euskadi.eus/es/tiendas-gourme...,tiendas,True


In [101]:
#eliminamos las columnas quality y euskal_sello
df_opendata = df_opendata.drop(columns=["quality", "euskal_sello"], errors="ignore")

In [102]:
df_opendata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 789 entries, 0 to 788
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   nombre       789 non-null    object 
 1   descripcion  780 non-null    object 
 2   municipio    789 non-null    object 
 3   provincia    789 non-null    object 
 4   latitud      789 non-null    float64
 5   longitud     789 non-null    float64
 6   type         787 non-null    object 
 7   tipo-comida  713 non-null    object 
 8   entorno      784 non-null    object 
 9   email        778 non-null    object 
 10  web          782 non-null    object 
 11  web_euskadi  789 non-null    object 
 12  categoria    789 non-null    object 
 13  calidad      789 non-null    bool   
dtypes: bool(1), float64(2), object(11)
memory usage: 81.0+ KB


In [105]:
# Asignar tipos correctos a todas las columnas
df_opendata["nombre"]     = df_opendata["nombre"].astype("string")
df_opendata["municipio"]  = df_opendata["municipio"].astype("string")
df_opendata["provincia"]  = df_opendata["provincia"].astype("string")
df_opendata["type"]       = df_opendata["type"].astype("string")
df_opendata["tipo-comida"]= df_opendata["tipo-comida"].astype("string")
df_opendata["entorno"]    = df_opendata["entorno"].astype("string")
# map(bool) convierte cualquier valor truthy/falsy antes del cast estricto a boolean
df_opendata["email"]      = df_opendata["email"].astype("string")
df_opendata["web"]        = df_opendata["web"].astype("string")
df_opendata["web_euskadi"]= df_opendata["web_euskadi"].astype("string")
df_opendata["categoria"]  = df_opendata["categoria"].astype("string")
df_opendata["calidad"]    = df_opendata["calidad"].astype("boolean")


In [107]:
df_opendata

,nombre,descripcion,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,email,web,web_euskadi,categoria,calidad
0,Agorregi,"El restaurante Agorregi, ubicado en el barrio ...",Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False
1,Aizian,Este moderno y acogedor restaurante fue diseña...,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False
2,Akelarre,Pedro Subijana desarrolla desde 1970 una cocin...,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
3,Alameda,La tercera generación es quien está a cargo de...,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
4,Almiketxu,Es un caserío típico vasco y está regentado po...,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
784,Ibáñez-Gozona,NaN,Tolosa,Gipuzkoa,43.139897,-2.073319,Denominación de Origen,<NA>,Montes y Valles vascos,<NA>,<NA>,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True
785,Saizar,NaN,Tolosa,Gipuzkoa,43.132487,-2.080295,Denominación de Origen,<NA>,Montes y Valles vascos,<NA>,<NA>,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True
786,Suna Gozotegia,NaN,Getaria,Gipuzkoa,43.302922,-2.205573,Denominación de Origen,<NA>,Costa Vasca,<NA>,<NA>,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True
787,Rafa Gorrotxategi,NaN,Tolosa,Gipuzkoa,43.126214,-2.083494,Denominación de Origen,<NA>,Montes y Valles vascos,info@rafagorrotxategi.eus,https://rafagorrotxategi.eus/museum/,https://turismoa.euskadi.eus/es/tiendas-gourme...,tiendas,True


# Enriquecer con los datos de Goolge Place

In [108]:

load_dotenv()
API_KEY = os.getenv("API_KEY_MAPS")

#print(API_KEY)

In [109]:
def search_place_id(nombre, municipio, lat, lng, web, api_key):
    """Busca el Place ID en Google Places. Primero por nombre, luego por web (fallback).
    Devuelve el place_id (str) o None si no se encuentra."""
    location_bias = {"circle": {"center": {"latitude": lat, "longitude": lng}, "radius": 500.0}}

    # Intento 1: por nombre + municipio
    r = requests.post(
        "https://places.googleapis.com/v1/places:searchText",
        json={"textQuery": f"{nombre}, {municipio}", "locationBias": location_bias},
        headers={"Content-Type": "application/json", "X-Goog-Api-Key": api_key, "X-Goog-FieldMask": "places.id"}
    )
    places = (safe_json(r, f"search '{nombre}'") or {}).get('places', [])

    # Intento 2 (fallback): por URL de la web
    if not places and web:
        print(f"[FALLBACK web] '{nombre}' → {web}")
        r2 = requests.post(
            "https://places.googleapis.com/v1/places:searchText",
            json={"textQuery": web, "locationBias": location_bias},
            headers={"Content-Type": "application/json", "X-Goog-Api-Key": api_key, "X-Goog-FieldMask": "places.id"}
        )
        places = (safe_json(r2, f"search web '{nombre}'") or {}).get('places', [])

    return places[0]['id'] if places else None


def get_place_details(place_id, api_key):
    """Obtiene rating, reseñas, precio, teléfono y foto de un Place ID.
    Devuelve el dict de detalles o None si falla."""
    r = requests.get(
        f"https://places.googleapis.com/v1/places/{place_id}",
        headers={
            "X-Goog-Api-Key": api_key,
            "X-Goog-FieldMask": "id,rating,userRatingCount,priceLevel,editorialSummary,nationalPhoneNumber,photos"
        }
    )
    return safe_json(r, f"details '{place_id}'")


def enrich_dataframe(df, api_key, limit=None):
    """Enriquece df_opendata con datos de Google Places API.
    limit=None procesa todas las filas; limit=N procesa solo las primeras N."""
    df["valoracion"]          = np.nan
    df["num_resenas"]         = np.nan
    df["nivel_precio"]        = np.nan
    df["url_imagen"]          = ""
    df["nationalPhoneNumber"] = ""

    subset = df.head(limit) if limit else df
    cont_ok, cont_ko = 0, 0

    for idx, row in subset.iterrows():
        nombre    = row["nombre"]
        municipio = row["municipio"]
        lat, lng  = row["latitud"], row["longitud"]
        web       = row["web"] if pd.notna(row["web"]) and str(row["web"]).strip() else None

        place_id = search_place_id(nombre, municipio, lat, lng, web, api_key)
        if not place_id:
            print(f"[SKIP] sin resultados para '{nombre}'")
            continue

        details = get_place_details(place_id, api_key)
        if not details:
            print(f"[SKIP] sin detalles para '{nombre}'")
            cont_ko += 1
            continue

        df.at[idx, "valoracion"]          = details.get("rating")
        df.at[idx, "num_resenas"]         = details.get("userRatingCount")
        df.at[idx, "nivel_precio"]        = details.get("priceLevel")
        df.at[idx, "nationalPhoneNumber"] = details.get("nationalPhoneNumber", "")

        # Photos API v1: el campo es photos[].name; la URL de media se construye así:
        # https://places.googleapis.com/v1/{photo_name}/media?maxWidthPx=800&key=API_KEY
        photos = details.get("photos", [])
        if photos:
            photo_name = photos[0].get("name", "")
            df.at[idx, "url_imagen"] = (
                f"https://places.googleapis.com/v1/{photo_name}/media?maxWidthPx=800&key={api_key}"
                if photo_name else ""
            )

        cont_ok += 1
        print(f"[OK] '{nombre}' → {details.get('rating')} ⭐ | {details.get('userRatingCount')} reseñas (ok: {cont_ok})")

    print(f"\n--- RESUMEN ---")
    print(f"Enriquecidos: {cont_ok} / {len(subset)} | Fallos en detalles: {cont_ko}")
    return df


# --- Ejecución (prueba con las 10 primeras filas) ---
df_opendata = enrich_dataframe(df_opendata, API_KEY)


C:\Users\jlalo\AppData\Local\Temp\ipykernel_16552\1657266180.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'PRICE_LEVEL_MODERATE' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "nivel_precio"]        = details.get("priceLevel")


[OK] 'Agorregi' → 4.6 ⭐ | 571 reseñas (ok: 1)
[OK] 'Aizian' → 4.7 ⭐ | 435 reseñas (ok: 2)
[OK] 'Akelarre' → 4.5 ⭐ | 1924 reseñas (ok: 3)
[OK] 'Alameda' → 4.6 ⭐ | 1098 reseñas (ok: 4)
[OK] 'Almiketxu' → 4.6 ⭐ | 1924 reseñas (ok: 5)
[OK] 'Ameztoi' → 4.5 ⭐ | 225 reseñas (ok: 6)
[OK] 'Andere' → 4.4 ⭐ | 767 reseñas (ok: 7)
[OK] 'Andra Mari' → 4.7 ⭐ | 1001 reseñas (ok: 8)
[OK] 'Arabako Txakolina, S.L.' → 4.5 ⭐ | 2 reseñas (ok: 9)
[OK] 'Aretxondo' → 4.4 ⭐ | 1059 reseñas (ok: 10)
[OK] 'Arkupe' → 4.3 ⭐ | 2022 reseñas (ok: 11)
[OK] 'Arlobi' → 4.5 ⭐ | 38 reseñas (ok: 12)
[OK] 'Asador Nicolás' → 4.5 ⭐ | 569 reseñas (ok: 13)
[OK] 'San Martin' → 4.6 ⭐ | 1735 reseñas (ok: 14)
[OK] 'Urkiola' → 4.5 ⭐ | 197 reseñas (ok: 15)
[OK] 'Astelena' → 4.6 ⭐ | 1589 reseñas (ok: 16)
[OK] 'Atalaia' → 4.6 ⭐ | 806 reseñas (ok: 17)
[OK] 'Azkue' → 4.2 ⭐ | 398 reseñas (ok: 18)
[OK] 'Azurmendi' → 4.7 ⭐ | 1899 reseñas (ok: 19)
[OK] 'Baserri Maitea' → 4.6 ⭐ | 1243 reseñas (ok: 20)
[OK] 'Bedua' → 4.3 ⭐ | 2081 reseñas (ok: 21

In [110]:
df_opendata


,nombre,descripcion,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,email,web,web_euskadi,categoria,calidad,valoracion,num_resenas,nivel_precio,url_imagen,nationalPhoneNumber
0,Agorregi,"El restaurante Agorregi, ubicado en el barrio ...",Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False,4.6,571.0,PRICE_LEVEL_MODERATE,https://places.googleapis.com/v1/places/ChIJFT...,943 22 43 28
1,Aizian,Este moderno y acogedor restaurante fue diseña...,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False,4.7,435.0,PRICE_LEVEL_MODERATE,https://places.googleapis.com/v1/places/ChIJde...,944 28 00 39
2,Akelarre,Pedro Subijana desarrolla desde 1970 una cocin...,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True,4.5,1924.0,PRICE_LEVEL_VERY_EXPENSIVE,https://places.googleapis.com/v1/places/ChIJXd...,943 31 12 09
3,Alameda,La tercera generación es quien está a cargo de...,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True,4.6,1098.0,PRICE_LEVEL_EXPENSIVE,https://places.googleapis.com/v1/places/ChIJfb...,943 64 27 89
4,Almiketxu,Es un caserío típico vasco y está regentado po...,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True,4.6,1924.0,PRICE_LEVEL_EXPENSIVE,https://places.googleapis.com/v1/places/ChIJZ5...,946 88 09 25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
784,Ibáñez-Gozona,NaN,Tolosa,Gipuzkoa,43.139897,-2.073319,Denominación de Origen,<NA>,Montes y Valles vascos,<NA>,<NA>,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True,4.7,155.0,PRICE_LEVEL_INEXPENSIVE,https://places.googleapis.com/v1/places/ChIJYw...,943 65 05 39
785,Saizar,NaN,Tolosa,Gipuzkoa,43.132487,-2.080295,Denominación de Origen,<NA>,Montes y Valles vascos,<NA>,<NA>,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True,4.5,42.0,PRICE_LEVEL_INEXPENSIVE,https://places.googleapis.com/v1/places/ChIJ7_...,943 67 16 49
786,Suna Gozotegia,NaN,Getaria,Gipuzkoa,43.302922,-2.205573,Denominación de Origen,<NA>,Costa Vasca,<NA>,<NA>,https://turismoa.euskadi.eus/es/pastelerias-y-...,tiendas,True,4.5,59.0,PRICE_LEVEL_INEXPENSIVE,https://places.googleapis.com/v1/places/ChIJ49...,943 14 08 12
787,Rafa Gorrotxategi,NaN,Tolosa,Gipuzkoa,43.126214,-2.083494,Denominación de Origen,<NA>,Montes y Valles vascos,info@rafagorrotxategi.eus,https://rafagorrotxategi.eus/museum/,https://turismoa.euskadi.eus/es/tiendas-gourme...,tiendas,True,4.4,320.0,PRICE_LEVEL_INEXPENSIVE,https://places.googleapis.com/v1/places/ChIJzw...,943 89 03 06


In [111]:
# guardamos los datos enriquecidos a CSV
df_opendata.to_csv("data/restaurantes_tiendas_enriquecido.csv", index=False)

In [ ]:
df_opendata.describe(include="all")


,nombre,municipio,provincia,latitud,longitud,organic,type,tipo-comida,entorno,web,categoria
count,774,774,774,774.000000,774.000000,772,772,714,769,774,774
unique,763,175,5,NaN,NaN,3,2,8,11,611,2
top,Txinparta,Donostia / San Sebastián,Gipuzkoa,NaN,NaN,0,Restauración,Restaurante,Montes y Valles vascos,,bodegas
freq,2,59,369,NaN,NaN,768,714,475,339,158,716
mean,NaN,NaN,NaN,43.131510,-2.455699,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,0.255014,0.392737,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,42.488302,-3.386813,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,43.034668,-2.799878,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,43.253863,-2.502752,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,43.307750,-2.095250,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
df_museos = pd.read_csv("data/Museos_patrimonio_limpio.csv")

In [39]:
df_museos

,Nombre,Descripción,Tipo de lugar,Dirección,Municipio,Provincia,Postal Code,Marcas,Email,Teléfono,WEB,URL amigable,Visita Guiada,Capacidad,Tienda,Tipo de Cultura,Coordenadas GPS,LATWGS84,LONWGS84
0,Museo Bibat,Ubicado en el Casco Antiguo de Vitoria-Gasteiz...,Museo,"Cuchillería, 54. Junto al Palacio de Bendaña.",Vitoria-Gasteiz,Araba/Álava,1001.0,Vitoria-Gasteiz,bibat@araba.eus,945 203 700,https://fourniermuseoabibat.eus/es/inicio,https://turismoa.euskadi.eus/es/museos/museo-b...,1.0,0.0,1.0,Otros,"42.8493094,-2.6711764000000358",42.849309,-2.671176
1,Museo de Faroles de la Cofradía de la Virgen B...,Este original museo alberga las 267 piezas de ...,Museo,"Zapatería, 33",Vitoria-Gasteiz,Araba/Álava,1001.0,Vitoria-Gasteiz,info@cofradiavirgenblanca.com,945 277 077,http://www.cofradiavirgenblanca.com/,https://turismoa.euskadi.eus/es/museos/museo-d...,1.0,100.0,0.0,Otros,"42.84809749,-2.67413986",42.848097,-2.674140
2,Museo de Ciencias Naturales de Álava,El Museo de Ciencias Naturales de Álava cuenta...,Museo,"Fundadora de las Siervas de Jesús, 24",Vitoria-Gasteiz,Araba/Álava,1001.0,Vitoria-Gasteiz,mcna@araba.eus,945 181 924,NaN,https://turismoa.euskadi.eus/es/museos/museo-d...,1.0,100.0,0.0,Ciencias Naturales,"42.8502903,-2.67462519",42.850290,-2.674625
3,Artium museoa,El/la visitante podrá encontrar en exposición ...,Museo,"Francia, 24",Vitoria-Gasteiz,Araba/Álava,1002.0,Vitoria-Gasteiz,museo@artium.eus,945 209 020,https://www.artium.eus/es/,https://turismoa.euskadi.eus/es/museos/artium-...,1.0,2892.0,1.0,Arte,"42.84942634,-2.66802071",42.849426,-2.668021
4,Museo de Armería,El Museo de Armería ofrece al visitante una ex...,Museo,"Paseo de Fray Francisco de Vitoria, 3",Vitoria-Gasteiz,Araba/Álava,1001.0,Vitoria-Gasteiz,museoarmeria@araba.eus,945 181 925,NaN,https://turismoa.euskadi.eus/es/museos/museo-d...,1.0,92.0,0.0,Otros,"42.8413475,-2.678544200000033",42.841347,-2.678544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
631,Galería J. M. Lumbreras,Esta galería posee tres salas de exposición y ...,Auditorios; Museos; Galerías de arte y salas d...,c/ Henao 3,Bilbao,Bizkaia,48009.0,Bilbao,galeria@galerialumbreras.com,944 244 545,http://galerialumbreras.com,https://turismoa.euskadi.eus/es/museos/galeria...,0.0,0.0,0.0,Patrimonio Cultural,"43.26396,-2.9294957",43.263960,-2.929496
632,Estudio Iñigo Manterola,"Iñigo Manterola es un artista polifacético, cu...",Auditorios; Museos; Galerías de arte y salas d...,"Polígono Errotaberri, Ola Bidea s/n",Zarautz,Gipuzkoa,20800.0,Costa Vasca,NaN,617 332 813,https://www.inigomanterola.com/,https://turismoa.euskadi.eus/es/museos/estudio...,0.0,0.0,0.0,Patrimonio Cultural,"43.27426,-2.1686642",43.274260,-2.168664
633,Castillo de Portilla,"Está ubicado en Portilla, concejo del municipi...",Patrimonio cultural,NaN,Zambrana,Araba/Álava,NaN,Montes y Valles vascos,NaN,NaN,https://turismo.euskadi.eus/es/patrimonio-cult...,https://turismoa.euskadi.eus/es/patrimonio-cul...,0.0,0.0,0.0,Patrimonio Cultural,"42.6703698,-2.8336449",42.670370,-2.833645
634,ABAO Bilbao Ópera,ABAO Bilbao Ópera es una institución cultural ...,Auditorios; Museos; Galerías de arte y salas d...,"José María Olavarri, 2",Bilbao,Bizkaia,NaN,Bilbao,abao@abao.org,NaN,https://www.abao.org/es,https://turismoa.euskadi.eus/es/museos/abao-bi...,0.0,0.0,0.0,Patrimonio Cultural,"43.2603473,-2.9274007",43.260347,-2.927401


In [ ]:
def search_place_id(Nombre, Municipio, LATWGS84, LONWGS84, WEB, api_key):
    """Busca el Place ID en Google Places. Primero por nombre, luego por web (fallback).
    Devuelve el place_id (str) o None si no se encuentra."""
    location_bias = {"circle": {"center": {"latitude": LATWGS84, "longitude": LONWGS84}, "radius": 500.0}}

    # Intento 1: por nombre + municipio
    r = requests.post(
        "https://places.googleapis.com/v1/places:searchText",
        json={"textQuery": f"{Nombre}, {Municipio}", "locationBias": location_bias},
        headers={"Content-Type": "application/json", "X-Goog-Api-Key": api_key, "X-Goog-FieldMask": "places.id"}
    )
    places = (safe_json(r, f"search '{Nombre}'") or {}).get('places', [])

    # Intento 2 (fallback): por URL de la web
    if not places and WEB:
        print(f"[FALLBACK web] '{Nombre}' → {WEB}")
        r2 = requests.post(
            "https://places.googleapis.com/v1/places:searchText",
            json={"textQuery": WEB, "locationBias": location_bias},
            headers={"Content-Type": "application/json", "X-Goog-Api-Key": api_key, "X-Goog-FieldMask": "places.id"}
        )
        places = (safe_json(r2, f"search web '{Nombre}'") or {}).get('places', [])

    return places[0]['id'] if places else None


def get_place_details(place_id, api_key):
    """Obtiene rating, reseñas, precio, teléfono y foto de un Place ID.
    Devuelve el dict de detalles o None si falla."""
    r = requests.get(
        f"https://places.googleapis.com/v1/places/{place_id}",
        headers={
            "X-Goog-Api-Key": api_key,
            "X-Goog-FieldMask": "id,rating,userRatingCount,priceLevel,editorialSummary,nationalPhoneNumber,photos"
        }
    )
    return safe_json(r, f"details '{place_id}'")


def enrich_dataframe(df, api_key, limit=None):
    """Enriquece df_opendata con datos de Google Places API.
    limit=None procesa todas las filas; limit=N procesa solo las primeras N."""
    df["valoracion"]          = np.nan
    df["num_resenas"]         = np.nan
    #df["nivel_precio"]        = np.nan
    df["url_imagen"]          = ""
    #df["nationalPhoneNumber"] = ""

    subset = df.head(limit) if limit else df
    cont_ok, cont_ko = 0, 0

    for idx, row in subset.iterrows():
        nombre    = row["Nombre"]
        municipio = row["Municipio"]
        lat, lng  = row["LATWGS84"], row["LONWGS84"]
        web       = row["WEB"] if pd.notna(row["WEB"]) and str(row["WEB"]).strip() else None

        place_id = search_place_id(nombre, municipio, lat, lng, web, api_key)
        if not place_id:
            print(f"[SKIP] sin resultados para '{nombre}'")
            continue

        details = get_place_details(place_id, api_key)
        if not details:
            print(f"[SKIP] sin detalles para '{nombre}'")
            cont_ko += 1
            continue

        df.at[idx, "valoracion"]          = details.get("rating")
        df.at[idx, "num_resenas"]         = details.get("userRatingCount")
        #df.at[idx, "nivel_precio"]        = details.get("priceLevel")
        #df.at[idx, "nationalPhoneNumber"] = details.get("nationalPhoneNumber", "")

        # Photos API v1: el campo es photos[].name; la URL de media se construye así:
        # https://places.googleapis.com/v1/{photo_name}/media?maxWidthPx=800&key=API_KEY
        photos = details.get("photos", [])
        if photos:
            photo_name = photos[0].get("name", "")
            df.at[idx, "url_imagen"] = (
                f"https://places.googleapis.com/v1/{photo_name}/media?maxWidthPx=800&key={api_key}"
                if photo_name else ""
            )

        cont_ok += 1
        print(f"[OK] '{nombre}' → {details.get('rating')} ⭐ | {details.get('userRatingCount')} reseñas (ok: {cont_ok})")

    print(f"\n--- RESUMEN ---")
    print(f"Enriquecidos: {cont_ok} / {len(subset)} | Fallos en detalles: {cont_ko}")
    return df

In [ ]:
df_museos = enrich_dataframe(df_museos, API_KEY, limit=10)


[OK] 'Museo Bibat' → 4.4 ⭐ | 242 reseñas (ok: 1)
[OK] 'Museo de Faroles de la Cofradía de la Virgen Blanca' → 4.7 ⭐ | 250 reseñas (ok: 2)
[OK] 'Museo de Ciencias Naturales de Álava' → 4.6 ⭐ | 770 reseñas (ok: 3)
[OK] 'Artium museoa' → 4.1 ⭐ | 2644 reseñas (ok: 4)
[OK] 'Museo de Armería' → 4.7 ⭐ | 729 reseñas (ok: 5)
[OK] 'Museo de Bellas Artes de Álava' → 4.6 ⭐ | 1250 reseñas (ok: 6)
[OK] 'Villa Lucía. Centro Temático del vino' → 4.6 ⭐ | 1737 reseñas (ok: 7)
[OK] 'Museo Diocesano de Arte Sacro' → 4.6 ⭐ | 144 reseñas (ok: 8)
[OK] 'Museo de la Policía Vasca' → 4.3 ⭐ | 49 reseñas (ok: 9)
[OK] 'Museo Etnográfico Usatxi de Pipaón' → 4.9 ⭐ | 27 reseñas (ok: 10)

--- RESUMEN ---
Enriquecidos: 10 / 10 | Fallos en detalles: 0
